# FATFORMER-XLA: PIPELINE TRÍCH XUẤT VÀ ĐÓNG GÓI COMMUNITYFORENSICS-EVAL (CVPR 2025)
### BỘ KIỂM ĐỊNH NGOẠI VI ĐỘC LẬP (UNSEEN EXTERNAL BENCHMARK — 2.000 ẢNH)

> **Mục tiêu**: Thu thập 2.000 ảnh từ `OwensLab/CommunityForensics-Eval` qua cơ chế Streaming Parquet (không tốn đĩa Colab).
> **Phân tầng 5 họ bộ sinh**: `1_commercial`, `2_latent_diffusion`, `3_pixel_diffusion`, `4_modern_gans`, `5_other_autoregressive`.
> **Bảo toàn Zero Leakage**: Kiểm toán cách ly SHA-256 đối chiếu `data_manifest.json` để loại trừ trùng lặp.
> **Lưu trữ**: Đóng gói thành `test_communityforensics.tar` (PNG Lossless) lưu tại `MyDrive/Fatformer/datasets/`.

### Bước 1: Kết nối Google Drive và Kiểm tra Dung Lượng Khả Dụng

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!df -h /content | tail -1   # Yêu cầu dung lượng trống >= 10 GB

### Bước 2: Cài Đặt Thư Viện Cần Thiết
Cài đặt thư viện `datasets` của Hugging Face để nạp dữ liệu dạng streaming.

In [ ]:
!pip install -q datasets "pillow<13.0,>=8.0"

### Bước 3: Trích Xuất Phân Tầng 2.000 Ảnh (Đã Fix Toàn Bộ Schema)
- **Split**: `CompEval` (chứa 51.836 ảnh).
- **Byte thô**: `image_data` qua `io.BytesIO` chuyển sang PIL Image.
- **Lọc kiến trúc**: Cột `architecture` ('Commercial', 'LatDiff', 'PixDiff', 'GAN', 'other').
- **Lọc nguồn ảnh thật**: Cột `real_source` ('coco', 'ffhq', 'raise', 'laion').
- **Đa dạng hóa**: Tối đa 60 ảnh cho mỗi tên mô hình cụ thể `model_name`.

In [ ]:
import os, io, json, hashlib, tarfile, gc
import pyarrow as pa
from collections import defaultdict
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm

TARGET_DIR = "/content/dataset_local/test_communityforensics"
DRIVE_TAR_PATH = "/content/drive/MyDrive/Fatformer/datasets/test_communityforensics.tar"
EXISTING_MANIFEST = "/content/drive/MyDrive/Fatformer/manifests/data_manifest.json"
CFE_MANIFEST_PATH = os.path.join(TARGET_DIR, "cfe_manifest.json")
SEED = 42

os.makedirs(TARGET_DIR, exist_ok=True)
os.makedirs(os.path.dirname(DRIVE_TAR_PATH), exist_ok=True)

# 1. Nạp danh sách SHA-256 cũ để đảm bảo Zero Data Leakage
existing_hashes = set()
if os.path.exists(EXISTING_MANIFEST):
    with open(EXISTING_MANIFEST, "r", encoding="utf-8") as f:
        manifest_data = json.load(f)
        records = manifest_data.get("records", manifest_data) if isinstance(manifest_data, dict) else manifest_data
        for entry in records:
            if "sha256" in entry:
                existing_hashes.add(entry["sha256"])
    print(f"[*] Đã nạp {len(existing_hashes)} mã băm SHA-256 từ data_manifest.json.")

# 2. Định nghĩa chỉ tiêu phân tầng 5 tập con (Cân bằng nhị phân 50/50)
QUOTAS = {
    "1_commercial": {
        "arch_key": "Commercial",
        "fake_max": 300,
        "max_per_model": 60,
        "real_quotas": {"coco": 150, "raise": 75, "laion": 75}
    },
    "2_latent_diffusion": {
        "arch_key": "LatDiff",
        "fake_max": 300,
        "max_per_model": 60,
        "real_quotas": {"coco": 150, "ffhq": 75, "laion": 75}
    },
    "3_pixel_diffusion": {
        "arch_key": "PixDiff",
        "fake_max": 200,
        "max_per_model": 200,  # Nới lỏng trần vì họ Pixel Diffusion chỉ có 1 mô hình (DeepFloyd IF)
        "real_quotas": {"coco": 100, "raise": 50, "ffhq": 50}
    },
    "4_modern_gans": {
        "arch_key": "GAN",
        "fake_max": 100,
        "max_per_model": 60,
        "real_quotas": {"ffhq": 50, "coco": 50}
    },
    "5_other_autoregressive": {
        "arch_key": "other",
        "fake_max": 100,
        "max_per_model": 100,  # Nới lỏng trần vì họ Autoregressive chỉ có 1 mô hình đại diện
        "real_quotas": {"raise": 50, "laion": 50}
    },
}

for group in QUOTAS:
    os.makedirs(f"{TARGET_DIR}/{group}/0_real", exist_ok=True)
    os.makedirs(f"{TARGET_DIR}/{group}/1_fake", exist_ok=True)

counts = {
    g: {
        "fake": 0,
        "real": {src: 0 for src in cfg["real_quotas"]},
        "models_collected": defaultdict(int)
    }
    for g, cfg in QUOTAS.items()
}

# 3. CƠ CHẾ RESUME THÔNG MINH: Nạp các ảnh đã tải thành công trước đó
manifest_records = []
if os.path.exists(CFE_MANIFEST_PATH):
    try:
        with open(CFE_MANIFEST_PATH, "r", encoding="utf-8") as f:
            manifest_records = json.load(f)
        for r in manifest_records:
            g = r["group"]
            lbl = r["label"]
            mname = r["model_name"]
            rsrc = r.get("real_source", "")
            hsh = r["sha256"]
            existing_hashes.add(hsh)
            if lbl == 1:
                counts[g]["fake"] += 1
                counts[g]["models_collected"][mname] += 1
            else:
                for k in counts[g]["real"]:
                    if k in rsrc:
                        counts[g]["real"][k] += 1
                        break
        print(f"[*] RESUME THÀNH CÔNG: Đã tìm thấy {len(manifest_records)} ảnh có sẵn. Tiếp tục thu thập...")
    except Exception as e:
        print(f"[!] Bắt đầu thu thập mới do: {e}")

total_collected = len(manifest_records)

print("[*] Đang kết nối Streaming Parquet từ OwensLab/CommunityForensics-Eval (split='CompEval')...")
# Không dùng shuffle buffer trong RAM để tránh memory drift
ds = load_dataset("OwensLab/CommunityForensics-Eval", split="CompEval", streaming=True)

pbar = tqdm(total=2000, initial=total_collected, desc="Đang trích xuất phân tầng 2.000 ảnh")

for item in ds:
    if total_collected >= 2000:
        break

    raw_bytes = item.get("image_data")
    if not raw_bytes:
        continue

    img_hash = hashlib.sha256(raw_bytes).hexdigest()
    if img_hash in existing_hashes:
        del raw_bytes
        continue

    label = int(item.get("label", -1))
    arch = str(item.get("architecture", "")).strip()
    model_name = str(item.get("model_name", "unknown")).strip()
    real_src = str(item.get("real_source", "")).lower().strip()

    assigned_group = None
    matched_r_src = None

    if label == 1:
        for g, cfg in QUOTAS.items():
            if arch.lower() == cfg["arch_key"].lower():
                if counts[g]["fake"] < cfg["fake_max"]:
                    max_allowed = cfg.get("max_per_model", 60)
                    if counts[g]["models_collected"][model_name] < max_allowed:
                        assigned_group = g
                        break
    elif label == 0:
        for g, cfg in QUOTAS.items():
            for src_key, src_max in cfg["real_quotas"].items():
                if src_key in real_src and counts[g]["real"][src_key] < src_max:
                    assigned_group = g
                    matched_r_src = src_key
                    break
            if assigned_group:
                break

    if assigned_group:
        try:
            img = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
        except Exception:
            del raw_bytes
            continue

        if label == 1:
            idx = counts[assigned_group]["fake"]
            safe_mname = model_name.replace("/", "_").replace(" ", "_")[:30]
            fname = f"fake_{idx:04d}_{safe_mname}.png"
            rel_path = f"{assigned_group}/1_fake/{fname}"
            counts[assigned_group]["fake"] += 1
            counts[assigned_group]["models_collected"][model_name] += 1
        else:
            idx = sum(counts[assigned_group]["real"].values())
            fname = f"real_{idx:04d}_{matched_r_src}.png"
            rel_path = f"{assigned_group}/0_real/{fname}"
            counts[assigned_group]["real"][matched_r_src] += 1

        save_path = os.path.join(TARGET_DIR, rel_path)
        img.save(save_path, format="PNG")
        existing_hashes.add(img_hash)

        manifest_records.append({
            "rel_path": rel_path,
            "group": assigned_group,
            "label": label,
            "architecture": arch,
            "model_name": model_name,
            "real_source": real_src,
            "sha256": img_hash
        })

        total_collected += 1
        pbar.update(1)

        del img

        # Dọn dẹp định kỳ mỗi 50 ảnh: Cả Python GC và PyArrow C++ Memory Pool
        if total_collected % 50 == 0:
            gc.collect()
            try:
                pa.default_memory_pool().release_unused()
            except Exception:
                pass
            # Ghi checkpoint cfe_manifest.json để đảm bảo an toàn tuyệt đối
            with open(CFE_MANIFEST_PATH, "w", encoding="utf-8") as f:
                json.dump(manifest_records, f, indent=2, ensure_ascii=False)

    del raw_bytes

pbar.close()
gc.collect()
try:
    pa.default_memory_pool().release_unused()
except Exception:
    pass

with open(CFE_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest_records, f, indent=2, ensure_ascii=False)

print(f"\n[✓] HOÀN TẤT XUẤT SẮC! Đã thu thập đủ {total_collected}/2.000 ảnh chuẩn hóa!")
print("[*] Thống kê chi tiết từng tập con:")
for g, cfg in QUOTAS.items():
    tot_r = sum(counts[g]['real'].values())
    exp_r = sum(cfg['real_quotas'].values())
    print(f"  • {g:<26}: Fake={counts[g]['fake']}/{cfg['fake_max']} | Real={tot_r}/{exp_r}")

### Bước 4: Đóng Gói Thành Tệp .TAR Đẩy Lên Google Drive
Sử dụng chuẩn `PAX_FORMAT` để tương thích 100% với môi trường Linux / Colab / Kaggle.

In [ ]:
print(f"[*] Đang đóng gói {TARGET_DIR} sang {DRIVE_TAR_PATH}...")
with tarfile.open(DRIVE_TAR_PATH, "w", format=tarfile.PAX_FORMAT) as tar:
    tar.add(TARGET_DIR, arcname="test_communityforensics")

tar_sz_gb = os.path.getsize(DRIVE_TAR_PATH) / (1024 ** 3)
print(f"[✓] ĐÓNG GÓI HOÀN TẤT! Kích thước file: {tar_sz_gb:.2f} GB")
print(f"[✓] Vị trí lưu trữ: {DRIVE_TAR_PATH}")

### Hoàn Tất Khâu Chuẩn Bị & Đóng Gói Dữ Liệu
> [✓] **Tệp lưu trữ vĩnh viễn**: `MyDrive/Fatformer/datasets/test_communityforensics.tar` đã được tạo thành công trên Google Drive.
> 🚀 **Bước tiếp theo**: Mở notebook chuyên dụng [`notebooks/eval_communityforensics.ipynb`](eval_communityforensics.ipynb) để kích hoạt GPU đo đạc Bảng A (Clean) & Bảng B (JPEG Q=30).